<a href="https://colab.research.google.com/github/slover1126/slover1126/blob/main/%ED%95%B8%EC%A6%88%EC%98%A8_%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D_7%EC%9E%A5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**7장 앙상블 학습과 랜덤 포레스트**



*   일련의 예측기로부터 예측을 수집한다면, 하나의 예측기보다 더 좋은 예측 결과를 얻을 수 있을 것임
*   이때 일련의 예측기들을 **앙상블(Ensemble)** 이라고 하기 때문에 위와 같은 방법을 **앙상블 학습(ensempble learning)** 이라고 하며, 앙상블 항습 알고리즘을 **앙상블 방법(ensemble method)** 라고 부름

*   예시로, 훈련 세트로부터 랜덤으로 서브셋을 분할해서, 일련의 결정트리를 학습시킨 다음, 개별 트리의 예측을 모아 가장 많은 선택을 받은 클래스를 앙상블의 예측으로 삼는 방법이 있음

*   이때, 결정 트리의 앙상블을 **랜덤 포레스트** 라고 함



**7.1 투표 기반 분류기**



*   더 좋은 분류기를 만드는 가장 간단한 방법은 각 분류기의 예측을 집계하는 것임
*   가장 많은 표를 얻은 클래스가 앙상블의 예측이 되느데, 이렇게 다수결로 정해지는 분류기를 **직접 투표(hard voting)** 분류기 라고  함

*   각 분류기가 **약한 학습기(weak learner)** (랜덤 추측보다 조금 더 높은 성능을 내는 분류기) 일지라도 그 수가 충분하게 많고 다양하다면 높은 정확도를 내는 **강한 학습기(strong learner)** 가 될 수 있음
*   이는 통계학의 **큰 수의 법칙(law of large numbers)** 에 의해 표본이 커질수록 실제 참값으로 수렴하기 때문임


*   다만 모든 분류기가 독립적이고 오차에 상관관계가 없다는 가정이 필요하기 때문에 다양한 알고리즘으로 학습시켜 앙상블 모델을 만드는것이 좋음(보통 같은 데이터셋으로 학습시켜서 이 가정이 깨짐)




In [1]:
#VotingClassifier 와 moons 데이터셋으로 투표기반 분류기 만들기
#VotingClassifier는 이름, 예측기 쌍을 넣으면 일반 분류기처럼 활용할 수 있음

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X , y = make_moons(n_samples = 500 , noise = 0.30 , random_state =42)
X_train, X_test, y_train, y_test = train_test_split(X,y, random_state=42)
voting_clf = VotingClassifier(
    estimators = [
        ('lr', LogisticRegression(random_state=42)),
        ('rf', RandomForestClassifier(random_state=42)),
        ('svc', SVC(random_state=42))
    ]
)

voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression(random_state=42)),
                             ('rf', RandomForestClassifier(random_state=42)),
                             ('svc', SVC(random_state=42))])

In [2]:
#VotingClassifier는 각 추정기를 복제하여 훈련시킴
#원본 추정기의 경우 estimators 속성을 통해 참조 가능하고, 훈련된 복제본은 estimators_ 속성에 저장됨
#리스트 대신 딕셔너리를 달하는 경우 named_estimators 나 named_estimators_ 를 사용할 수 있음
print(voting_clf.estimators) #내가 넘겨준 모델과 이름을 튜플로 반환
print(voting_clf.estimators_) #학습이 끝난 모델(복제된거)을 반환
print(voting_clf.named_estimators) #딕셔너리 형태로 이름과 원본 모델 반환
print(voting_clf.named_estimators_) #딕셔너리 형태로 이름과 복제하여 학습된 모델 반환

[('lr', LogisticRegression(random_state=42)), ('rf', RandomForestClassifier(random_state=42)), ('svc', SVC(random_state=42))]
[LogisticRegression(random_state=42), RandomForestClassifier(random_state=42), SVC(random_state=42)]
{'lr': LogisticRegression(random_state=42), 'rf': RandomForestClassifier(random_state=42), 'svc': SVC(random_state=42)}
{'lr': LogisticRegression(random_state=42), 'rf': RandomForestClassifier(random_state=42), 'svc': SVC(random_state=42)}


In [3]:
#테스트 세트에서 훈련된 개별 분류기의 정확도
for name, clf in voting_clf.named_estimators_.items():
    print(name, '=', clf.score(X_test, y_test))

lr = 0.864
rf = 0.896
svc = 0.896


In [4]:
#투표 기반 분류기의 pridct() 메서드로 직접 투표를 수행할 수 있음(테스트 세트의 첫번째 샘플에 대한 예측값 출력)
voting_clf.predict(X_test[:1])
# 1번 클래스로 예측함

array([1])

In [5]:
#그 이유는 3개의 분류기 중에 2개가 1번 클래스라고 예측했기 때문임
[clf.predict(X_test[:1]) for clf in voting_clf.estimators_]
# 순서대로 1번 분류기, 2번 분류기, 3번 분류기의 클래스 예측 결과임

[array([1]), array([1]), array([0])]

In [6]:
#테스트 세트에서 투표 기반 분류기의 정확도
voting_clf.score(X_test, y_test)
#개별 분류기보다 투표 기반 분류기가 성능이 좀 더 높은것을 확인할 수 있음

0.912



*   모든 투표에 참여한 분류기가 **클래스의 확률을 예측할 수 있다면,** (perdict_proba 메서드가 있다면) 개별 분류기의 확률을 평균 내어서 확률이 가장 높은 클래스를 예측할 수 있음
*   이러한 방식을 **간접 투표** 라고 하며, 확률이 높은 투표에 더 비중을 두기 때문에 직접 투표보다 성능이 높음
*   강하게 확신하는 모델에 대해 더 큰 가중치가 실리기 때문임 (직접 투표의 경우 확신하는 정도가 51%인 클래스에 대해서도 딱 하나의 클래스로만 반영하여 투표하기 때문에)


In [7]:
#간접 투표로 바꿔보기
voting_clf.voting = 'soft'
voting_clf.named_estimators['svc'].probability = True
voting_clf.fit(X_train, y_train)
voting_clf.score(X_test, y_test)
# 0.92로 성능이 상승했음을 알 수 있음

0.92

**7.2 배깅과 페이스팅**



*   다양한 분류기를 만드는 방법은 앞서 한것과 같이 여러 알고리즘을 사용할 수 있음
*   다른 방법으로는 같은 알고리즘을 쓰고 훈련 세트의 서브셋을 랜덤으로 구성하여 분류기를 다르게 학습시키는 방법이 있음


*   이때, 서브셋을 구성할 때 중복을 허용해서 뽑으면 **배깅(bagging, bootstrap aggregating)**, 중복을 허용하지 않고 뽑으면 **페이스팅(pasting)** 이라고 함
*   배깅과 페이스팅은 하나의 전체 훈련 샘플로 여러 예측기에 걸쳐 사용할 수 있지만, 배깅만이 한 예측기를 위해 같은 훈련 샘플을 여러번 샘플링 할 수 있음






*   모든 예측기가 훈련을 마치면 앙상블은 모든 예측기의 예측을 모아 새로운 샘플에 대한 예측을 만들어냄
*   일반적으로 분류는 **통계적 최빈값**을, 회귀에 대해서는 평균을 계산함(집계함수)


*   개별 예측기는 원본 데이터를 다 써서 훈련한 경우보다 편향되어 있지만(모델의 복잡도가 단순하지만) 최빈값과 평균을 구해서 계산하면(집계함수를 거치면) 편향과 분산이 모두 감소하는 효과가 있음
*   일반적으로 앙상블의 결과는 원본 데이터셋으로 하나의 예측기를 훈려시킬 때와 비교해서 편향은 비슷하지만 분산은 감소함
*   예측기는 동시에 다른 CPU 코어나 서버에서 병렬로 학습시킬 수 있고, 예측도 병렬로 수행 가능





**7.2.1 사이킷런의 배깅과 페이스팅**



*   사이킷런은 **BaggingClassifier**(분류)와 **BaggingRegressor**(회귀) 클래스를 이용할 수 있음



In [8]:
#결정 트리 500개의 앙상블을 학습시키는 코드
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(DecisionTreeClassifier(),
                            n_estimators = 500, # 생성할 트리의 수를 정함
                            max_samples = 100, #전체 훈련 세트에서 중복을 허용해서 100개를 샘플링
                           n_jobs = -1, #훈련과 예측에 사용 가능한 CPU 코어 수를 지정(-1 이면 모든 코어 사용)
                            random_state = 42) # 재현성 고정

bag_clf.fit(X_train, y_train)
#BaggingClassifier는 기반이 되는 분류기가 결정트리 분류기처럼 클래스 확률을 추정할 수 있다면(predict_proba 함수가 있으면) 자동으로 간접투표 방식을 사용함

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=100,
                  n_estimators=500, n_jobs=-1, random_state=42)



*   **n_estimators**: 생성할 모델 수 지정
*   **max_samples**: 훈련 세트에서 중복을 허용하여 랜덤으로 뽑을 수 (배깅의 경우이고, 페이스팅 이용시 **bootstrap = False** 로 지정)
*   **n_jobs**: 훈련과 예측에 사용 가능한 CPU 코어 수를 지정(-1 이면 모든 코어 사용)
* **random_state** : 재현성 고정





*   배깅의 경우 학습하는 서브셋에 다양성을 추가하여 페이스팅보다 편향이 조금 더 높음(전체 데이터를 다 보지 못해서 모델이 과소적합될 우려가 있음)
*   그렇지만 다양성을 추가하여 예측기 간의 상관관계를 줄여 앙상블의 분산이 줄어듦(데이터를 다 쓰면 사실상 같은 분류기 100개로 예측하는 효과가 있어 앙상블 효과가 감소, 배깅을 쓰면 어떤 트리는 A 데이터를 보지 못하고 어떤 트리는 B데이터를 보지 못해서 트리가 달라짐->상관관계가 줄어드는 효과가 발생함, 노이즈에 대한 오차들이 평균내거나 최빈값을 통해 상쇄되면서 분산이 줄어듦)
*    전반적으로 배깅이 선호되지만 시간과 CPU 파워에 여유가 있다면 배깅과 페이스팅을 모두 평가하여 더 나은쪽을 선택하는 것이 좋음



**7.2.2 OOB 평가**


*   BaggingClassifier는 bootstrap=True인 경우 전체 훈련세트에 대해 복원추출로 랜덤 서브셋을 구성함
*   이는 평균적으로 전체 훈련 세트의 **약 63% 정도**만 샘플링이 됨

* 이때 선택되지 않은 나머지 **37% 정도를 OOB(out of bag) 샘플** 이라고 부름(전체 훈련 데이터셋(크기 $m$)에서 독립적으로 복원추출을 새로 수행하기 때문에 예측기마다 남겨진 37%는 모두 다름, 왜 이 숫자인지는 교재 P.276 참조)

*   훈련시 이 OOB 샘플을 사용하지 않으므로 이를 이용하여 평가를 진행할 수 있음


*   앙상블의 평가는 각 예측기의 OOB 평가를 평균하여 얻음(자신을 한번도 쓰지 않은 모델 집합에 투입하여 나온 예측값들의 평균을 내서 실제 정답과 비교해 OOB 스코어를 연산)



In [9]:
#BaggingClassifier에서 oob_score = True로 설정하면 훈련이 끝난 후 자동으로 OOB 평가를 수행함
bag_clf = BaggingClassifier(DecisionTreeClassifier(), n_estimators = 500, oob_score=True, n_jobs=-1, random_state=42)

bag_clf.fit(X_train, y_train)
bag_clf.oob_score_ #평가 점수 결과는 .oob_score_에 저장되어 있음
#이 BaggingClassifier는 테스트 세트에서 89.6% 정도의 정확도를 얻을 것으로 예측 가능

0.896

In [10]:
#실제 테스트 세트로 확인하기
from sklearn.metrics import accuracy_score

y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)
# 테스트 세트에서 약 92%의 정확도를 얻음을 확인

0.92

In [15]:
#OOB 샘플에 대한 결정 함수의 값도 .oob_decision_function_으로 확인 가능
bag_clf.oob_decision_function_[:3] #상위 3개값 출력
#기반이 되는 예측기가 predict_proba() 메서드를 가지고 있어 결정 함수는 각 훈련 샘플의 클래스 확률 반환
#1번 샘플의 경우 0번 클래스일 확률이 0.32352941, 1번 클래스일 확률이 0.67647059 이라 예측

array([[0.32352941, 0.67647059],
       [0.3375    , 0.6625    ],
       [1.        , 0.        ]])

**7.3 랜덤 패치와 랜덤 서브스페이스**



*   BaggingClassifier는 특성에 대한 샘플링도 지원함 **max_features** 와 **bootstrap_features** 로 조절
*  BaggingClassifier에서 **max_features**는 개별 분류기마다 max_features에 할당된 비율만큼 특성을 뽑고, 그 특성으로만 그 분류기를 학습함


*   **bootstrap_features** 는 그 특성을 뽑을 때 부트스트랩을 적용할지 결정함(BaggingClassifier에서 기본은 False)
*   훈련 특성과 샘플을 모두 샘플링하면 **랜덤 패치 방식(random patches method)**, 훈련 샘플을 모두 사용하고(bootstrap = False이고 max_samples=1.0), 특성을 샘플링하는 방식(bootstrap_features=True 또는 max_features를 1보다 작게 설정)을 **랜덤 서브스페이스 방식(random subspaces method)** 라고 함
*   이 방법은 고차원 데이터를 다룰 때 유용함



**7.4 랜덤 포레스트**



*   랜덤 포레스트는 결정 트리의 앙상블로 BaggingClassifier/Regressor와 다르게 각 노드가 쪼개질 때마다 특성을 랜덤으로 추출하여 분류나 회귀를 진행함
*  **RandomForestClassifier** 와 **RandomForestRegressor** 가 있으며 일반적으로 각 개별 트리의 훈련 세트는 **max_samples**로 정해짐





In [16]:
#최대 16개의 리프 노드를 갖는 500개의 결정 트리로 이루어진 랜덤포레스트 분류기

from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes = 16, n_jobs = -1, random_state=42)

rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)




*   몇가지 예외가 있긴 하지만 트리의 성장을 조절하기 위한 DecisionTreeClassifier의 하이퍼파라미터와 BaggingClassifier의 하이퍼파라미터를 모두 가지고 있음
*   앞서 적은것과 같이 랜덤 포레스트 알고리즘은 트리의 노드를 분할할 때 전체 특성 중에서 최선의 특성을 찾는 것이 아닌 **랜덤으로 선택된 특성 후보 중에서 최적의 특성을 찾는 식으로 무작위성을 더 높임**


*   RandomForestClassifier는 기본적으로 $\sqrt{n}$ 개의 특성을 선택($n$은 전체 특성의 개수, RandomForestRegressor는 기본적으로 모든 특성 다 사용)
*   이는 결국 트리를 다양하게 만들고 편향을 손해보는 대신(가장 최적의 분할 특성이 뽑히지 않는 경우도 있어 후순위의 특성이 선택될 수 있으므로) 분산을 낮추어 전체적으로 더 좋은 모델을 만듦






**7.4.1 엑스트라 트리**



*   일반적으로 랜덤포레스트는 특성들의 랜덤 서브셋을 구성하여 그 중에서 최적의 특성과 임계점을 찾아 노드를 분할함

*   그러나 그 대신 **트리를 더욱 랜덤하게 만들기 위해서 후보 특성들을 사용하여 랜덤으로 노드 안의 샘플을 분할 한 후, 그중에서 최적의 분할을 선택하는 방식**도 고려할 수 있음

*   이렇게 하려면 RandomForestClassifier를 만들 때 splitter="random" 으로 지정하면 됨



*   이러한 극단적으로 랜덤한 트리의 랜덤 포레스트를 **익스트림 랜덤 트리(extremely randomized tree)** 앙상블 혹은 **엑스트라 트리(extra-tree)** 라고 함
*   **사이킷런의 ExtraTreesClassifier/Regressor를 통해 구현할 수 있음**(bootstrap 하이퍼파라미터가 기본적으로 False인걸 제외하면 사용법은 RandomForestClassifier/Regressor와 같음)

